In [2]:
from collections import Counter
from fractions import Fraction
import math

def calculate_bleu_detailed(reference, candidate):
    """Calculate BLEU score between reference and candidate sentences."""
    print(f"original reference: {reference}, and original candidate: {candidate}.")
    # Convert strings to lists of words.
    reference = [reference.split()]  # Wrap in list since we expect list of references.
    candidate = candidate.split()
    print(f"reference: {reference}, and candidate: {candidate}.")
    
    # Use default weights for BLEU-4 (equal weights).
    weights = (0.25, 0.25, 0.25, 0.25)
    
    # Calculate modified n-gram precisions for n=1,2,3,4.
    precisions = []
    for n in range(1, 5):
        # Get n-grams for candidate and reference.
        candidate_ngrams = Counter([tuple(candidate[i:i+n]) 
                                  for i in range(len(candidate)-n+1)]) if len(candidate) >= n else Counter()
        print(f"n-gram: {n}, candidate n-grams: {candidate_ngrams}.")
        
        # Find max reference counts.
        max_ref_counts = {}
        for ref in reference:
            ref_ngrams = Counter([tuple(ref[i:i+n]) 
                                for i in range(len(ref)-n+1)]) if len(ref) >= n else Counter()
            for ngram in candidate_ngrams:
                max_ref_counts[ngram] = max(max_ref_counts.get(ngram, 0), ref_ngrams[ngram])
            print(f"reference n-grams: {ref_ngrams}.")
            print(f"max reference counts: {max_ref_counts}.")
        
        # Calculate clipped counts.
        clipped_counts = {ngram: min(count, max_ref_counts.get(ngram, 0))
                         for ngram, count in candidate_ngrams.items()}
        numerator = sum(clipped_counts.values())
        denominator = sum(candidate_ngrams.values()) if candidate_ngrams else 1
        precisions.append(0.1/denominator if not numerator else( 1.0*numerator / denominator if denominator else 0))
        print(f"clipped counts: {clipped_counts}, precision: {precisions[-1]}.")

    # Calculate brevity penalty.
    ref_len = len(reference[0])  # Use first reference length.
    cand_len = len(candidate)
    print(f"reference length: {ref_len}, candidate length: {cand_len}.")

    bp = math.exp(1 - ref_len/cand_len) if cand_len < ref_len else 1
    # Calculate final BLEU score.
    if precisions[0] == 0:
        return 0
    
    log_precisions = [math.log(p) if p > 0 else 0.1 for p in precisions]
    weighted_score = sum(w * p for w, p in zip(weights, log_precisions))
    bleu = bp * math.exp(weighted_score)
    print(f"brevity penalty: {bp}, log precisions: {log_precisions}, weighted score: {weighted_score}, BLEU score: {bleu}.")
    return bleu

In [3]:
reference_sentence = "The cat is on the mat"
candidate_sentence = "The cat is on the mat"
bleu_score = calculate_bleu_detailed(reference_sentence, candidate_sentence)
print(f"BLEU score: {bleu_score}")

original reference: The cat is on the mat, and original candidate: The cat is on the mat.
reference: [['The', 'cat', 'is', 'on', 'the', 'mat']], and candidate: ['The', 'cat', 'is', 'on', 'the', 'mat'].
n-gram: 1, candidate n-grams: Counter({('The',): 1, ('cat',): 1, ('is',): 1, ('on',): 1, ('the',): 1, ('mat',): 1}).
reference n-grams: Counter({('The',): 1, ('cat',): 1, ('is',): 1, ('on',): 1, ('the',): 1, ('mat',): 1}).
max reference counts: {('The',): 1, ('cat',): 1, ('is',): 1, ('on',): 1, ('the',): 1, ('mat',): 1}.
clipped counts: {('The',): 1, ('cat',): 1, ('is',): 1, ('on',): 1, ('the',): 1, ('mat',): 1}, precision: 1.0.
n-gram: 2, candidate n-grams: Counter({('The', 'cat'): 1, ('cat', 'is'): 1, ('is', 'on'): 1, ('on', 'the'): 1, ('the', 'mat'): 1}).
reference n-grams: Counter({('The', 'cat'): 1, ('cat', 'is'): 1, ('is', 'on'): 1, ('on', 'the'): 1, ('the', 'mat'): 1}).
max reference counts: {('The', 'cat'): 1, ('cat', 'is'): 1, ('is', 'on'): 1, ('on', 'the'): 1, ('the', 'mat'): 1

In [4]:
candidate_sentence = "The cat is setting on the mat"
bleu_score = calculate_bleu_detailed(reference_sentence, candidate_sentence)
print(f"BLEU score: {bleu_score}")

original reference: The cat is on the mat, and original candidate: The cat is setting on the mat.
reference: [['The', 'cat', 'is', 'on', 'the', 'mat']], and candidate: ['The', 'cat', 'is', 'setting', 'on', 'the', 'mat'].
n-gram: 1, candidate n-grams: Counter({('The',): 1, ('cat',): 1, ('is',): 1, ('setting',): 1, ('on',): 1, ('the',): 1, ('mat',): 1}).
reference n-grams: Counter({('The',): 1, ('cat',): 1, ('is',): 1, ('on',): 1, ('the',): 1, ('mat',): 1}).
max reference counts: {('The',): 1, ('cat',): 1, ('is',): 1, ('setting',): 0, ('on',): 1, ('the',): 1, ('mat',): 1}.
clipped counts: {('The',): 1, ('cat',): 1, ('is',): 1, ('setting',): 0, ('on',): 1, ('the',): 1, ('mat',): 1}, precision: 0.8571428571428571.
n-gram: 2, candidate n-grams: Counter({('The', 'cat'): 1, ('cat', 'is'): 1, ('is', 'setting'): 1, ('setting', 'on'): 1, ('on', 'the'): 1, ('the', 'mat'): 1}).
reference n-grams: Counter({('The', 'cat'): 1, ('cat', 'is'): 1, ('is', 'on'): 1, ('on', 'the'): 1, ('the', 'mat'): 1}).
